# Quem Resolve? · Assistente de voz (GS 2026.2 · PLN)
Fluxo: **microfone → STT → guardrail → RAG → LLM → adaptação para fala → TTS → alto-falante**.

**Antes de tudo:** menu `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`.
Rode as células **de cima para baixo**. O passo a passo completo está em `GUIA_EXECUCAO.md`.

## Etapa 1 · Projeto e dependências (≈ 5 min)
Envie o `quem-resolve-voz.zip` (já com os áudios gravados dentro) pelo ícone de pasta à esquerda, ou preencha a URL do GitHub.

In [ ]:
import os
URL_GITHUB = ""  # ex.: "https://github.com/usuario/quem-resolve-voz.git" (deixe vazio para usar o zip)

if os.path.exists("/content/quem-resolve-voz.zip"):
    !unzip -oq /content/quem-resolve-voz.zip -d /content
elif URL_GITHUB:
    !git clone -q {URL_GITHUB} /content/quem-resolve-voz
else:
    raise SystemExit("Envie o quem-resolve-voz.zip para /content ou preencha URL_GITHUB.")
%cd /content/quem-resolve-voz
!ls

In [ ]:
!pip install -q -r requirements.txt

## Etapa 2 · Chaves (≈ 1 min)
No ícone de **chave** (Secrets) à esquerda, crie e libere para este notebook:
- `LLM_API_KEY`: chave da Groq (console.groq.com → API Keys). **Obrigatória.**
- `APP_SENHA`: senha para proteger o link público da interface. Opcional, mas recomendada.

In [ ]:
import os
from google.colab import userdata
os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
try:
    os.environ["APP_SENHA"] = userdata.get("APP_SENHA"); os.environ["APP_USUARIO"] = "grupo"
except Exception:
    print("APP_SENHA não definida: o link público ficará sem senha.")
os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["LLM_MODELO"] = "openai/gpt-oss-120b"
os.environ["STT_MODELO"] = "small"
os.environ["TTS_MOTOR"] = "piper"
os.environ["CONTATO_GRUPO"] = "seu-email@exemplo.com"  # aparece na tela de consentimento

In [ ]:
# Confere se os modelos configurados ainda existem na Groq (eles são descontinuados com frequência)
from pipelines.voz.llm import listar_modelos
disponiveis = listar_modelos(os.environ["LLM_BASE_URL"], os.environ["LLM_API_KEY"])
for m in ["openai/gpt-oss-120b", "openai/gpt-oss-20b"]:
    print(("OK      " if m in disponiveis else "FALTANDO"), m)
print("\nOutros modelos de texto disponíveis:", [m for m in disponiveis if "whisper" not in m and "guard" not in m])

## Etapa 3 · Conferências rápidas (≈ 3 min)
Testes automatizados (devem dar **102 passed**) e uma conversa de 3 turnos por texto, com áudio.

In [ ]:
!pytest testes/unitarios -q

In [ ]:
from pipelines.voz.tts import baixar_voz_piper
from pipelines.voz.config import Config
baixar_voz_piper(Config().piper_voz)

from IPython.display import Audio, display
from pipelines.voz import AssistenteVoz, Memoria
assistente = AssistenteVoz()
mem = Memoria()
for pergunta in ["Tem um buraco enorme na minha rua.", "E como eu faço para registrar isso?", "Me passa uma receita de bolo?"]:
    r = assistente.processar_texto(pergunta, mem)
    print("Cidadão:", pergunta); print("Assistente:", r.resposta_texto); print("Falado:", r.resposta_falada); print(r.latencias, "\n")
    display(Audio(r.audio_resposta))

## Etapa 4 · Testes e métricas
Cada célula grava os resultados em `testes/resultados/`. Copie as tabelas impressas para o relatório.

In [ ]:
# 4.1 Calibração do RAG (≈ 1 min). Se o limiar sugerido for diferente de 0.35, ajuste rag_limiar em pipelines/voz/config.py
!python testes/calibrar_rag.py

In [ ]:
# 4.2 STT: WER nos 20 áudios, Whisper small x medium (≈ 5 min)
!python testes/avaliar_stt.py --modelos small medium

In [ ]:
# 4.3 LLM: 30 perguntas, gpt-oss-20b x gpt-oss-120b, com LLM juiz (≈ 8 min).
# Se aparecer erro 429 (limite da Groq), espere alguns minutos ou rode no dia seguinte.
!python testes/avaliar_llm.py --modelos openai/gpt-oss-20b openai/gpt-oss-120b --juiz openai/gpt-oss-120b

In [ ]:
# 4.4 TTS: gera os 20 áudios (10 Piper + 10 edge-tts) e o formulário dos avaliadores (≈ 2 min)
!python testes/avaliar_tts.py gerar --motores piper edge
from google.colab import files; files.download("testes/tts/formulario_avaliacao.html")

In [ ]:
# 4.5 Ponta a ponta: 10 conversas x 3 turnos. A fala do cidadão é gerada por voz sintética (≈ 5 min)
!python testes/avaliar_e2e.py

In [ ]:
# 4.6 Baixar tudo: resultados + áudios do TTS + planilha de avaliação
!zip -qr resultados.zip testes/resultados testes/tts logs
from google.colab import files; files.download("resultados.zip")

### Depois que as 3 pessoas avaliarem o TTS
Cada avaliador manda de volta o texto gerado pelo formulário. Salve cada um como `avaliador_1.csv`, `avaliador_2.csv` e `avaliador_3.csv`, envie para a pasta `testes/tts/notas/` (ícone de pasta) e rode:

In [ ]:
!python testes/avaliar_tts.py resumir

## Etapa 5 · Interface para o vídeo
Abra o link `*.gradio.live`, entre com usuário `grupo` e a `APP_SENHA`, aceite o aviso e converse. Para encerrar, pare a célula (■).

In [ ]:
!python app_gradio.py --share